# Обработка и подготовка данных с помощью Tidyverse

Многоязычный конвейер данных: Python загружает данные → R обрабатывает с помощью dplyr → Python визуализирует результаты.

Демонстрирует работу **SharedVFS** — общей файловой системы, позволяющей Python и R обмениваться файлами.

## 1. Python: Загрузка набора данных

In [ ]:
import micropip
await micropip.install('pandas')
import pandas as pd, pyodide.http, os

url = "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv"
resp = await pyodide.http.pyfetch(url)
text = await resp.string()

os.makedirs("/shared/data", exist_ok=True)
with open("/shared/data/gapminder.csv", "w") as f:
    f.write(text)

df = pd.read_csv("/shared/data/gapminder.csv")
print(f"Загружено: {df.shape[0]} строк, {df.shape[1]} столбцов")
df.head()

## 2. R: Установка dplyr + tidyr и чтение общих данных

In [ ]:
install.packages(c("dplyr", "tidyr"))
library(dplyr)

gap <- read.csv("/shared/data/gapminder.csv")
cat("Прочитано из SharedVFS:", nrow(gap), "строк\n")
glimpse(gap)

## 3. dplyr: Сводные показатели по континентам (2007)

In [ ]:
gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    countries = n(),
    mean_life = round(mean(lifeExp), 1),
    median_gdp = round(median(gdpPercap), 0),
    total_pop = sum(as.numeric(pop))
  ) %>%
  arrange(desc(mean_life))

## 4. dplyr: Наибольший прирост продолжительности жизни

In [ ]:
gains <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  select(country, continent, year, lifeExp) %>%
  tidyr::pivot_wider(names_from = year, values_from = lifeExp,
                     names_prefix = "y") %>%
  mutate(gain = y2007 - y1952) %>%
  arrange(desc(gain)) %>%
  head(10)
gains

## 5. dplyr: Рост населения по континентам

In [ ]:
pop_growth <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  group_by(continent, year) %>%
  summarize(total_pop = sum(as.numeric(pop)), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = year, values_from = total_pop,
                     names_prefix = "pop_") %>%
  mutate(growth_pct = round((pop_2007 / pop_1952 - 1) * 100, 1)) %>%
  arrange(desc(growth_pct))
pop_growth

## 6. R: Запись результатов в SharedVFS

In [ ]:
# Запись сводки по континентам для визуализации в Python
summary_2007 <- gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    mean_life = round(mean(lifeExp), 1),
    mean_gdp = round(mean(gdpPercap), 0),
    .groups = "drop"
  )
write.csv(summary_2007, "/shared/data/r_summary.csv", row.names = FALSE)
cat("Записано в /shared/data/r_summary.csv\n")
summary_2007

## 7. Python: Визуализация результатов обработки R

In [ ]:
import micropip
await micropip.install('plotly')
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

r_summary = pd.read_csv("/shared/data/r_summary.csv")
print("Прочитано из SharedVFS (записано R):")
print(r_summary.to_string(index=False))

fig = px.bar(r_summary, x="continent", y="mean_life",
             title="Средняя продолжительность жизни по континентам (2007) — из R → Python",
             labels={"mean_life": "Ожидаемая продолжительность жизни (лет)", "continent": "Континент"},
             color="continent")
fig.update_layout(template='plotly_dark', showlegend=False)
show_plotly(fig)

In [ ]:
fig = px.scatter(r_summary, x="mean_gdp", y="mean_life",
                 text="continent", size=[40]*len(r_summary),
                 title="ВВП против продолжительности жизни по континентам (сводка R → график Python)",
                 labels={"mean_gdp": "Средний ВВП на душу населения", "mean_life": "Средняя продолжительность жизни"})
fig.update_traces(textposition="top center")
fig.update_layout(template='plotly_dark')
fig.update_yaxes(range=[r_summary['mean_life'].min() - 2, r_summary['mean_life'].max() + 6])
show_plotly(fig)

## Ключевые итоги

- **Python** загрузил данные CSV в каталог `/shared/data/`
- **R** прочитал данные через SharedVFS и обработал их с помощью конвейеров dplyr
- **R** записал сформированный отчет обратно в `/shared/data/r_summary.csv`
- **Python** считал вывод R и построил интерактивные графики с помощью Plotly

Весь обмен файлами происходит через SharedVFS — без необходимости вручную импортировать или экспортировать файлы.